# **Fase 4. 2. Diseño y construcción de dashboard de inteligencia de negocios** #
Oscar Eduardo Ladino Rey

Métodos Cuantitativos y Cualitativos para los negocios

Maestria en Administración de Organizaciones

UNAD

**ALISTAMIENTO DEL ENTORNO E IMPORTACIÓN DE LIBRERÍAS**

In [ ]:
import pandas as pd
import requests

# 1. Definir la URL del recurso de la API (BDUA Régimen Subsidiado)
api_url = "https://www.datos.gov.co/resource/d7a5-cnra.json"

# 2. Configurar los parámetros de la consulta
# Usamos $limit para aumentar la cantidad de registros (ajusta este valor según la RAM de tu entorno)
parametros = {
    "$limit": 50000
}

# 3. Ejecutar la petición GET
print("Conectando a la API de BDUA...")
respuesta = requests.get(api_url, params=parametros)

# 4. Validar la conexión y crear el DataFrame
if respuesta.status_code == 200:
    print("¡Conexión exitosa! Cargando datos en Pandas...")
    datos_json = respuesta.json()

    # Transformar el JSON a un DataFrame
    df_bdua = pd.DataFrame(datos_json)

    print(f"Dataset cargado con éxito: {df_bdua.shape[0]} filas y {df_bdua.shape[1]} columnas.")
    display(df_bdua.head())
else:
    print(f"Error en la conexión. Código HTTP: {respuesta.status_code}")

Conectando a la API de BDUA...
¡Conexión exitosa! Cargando datos en Pandas...
Dataset cargado con éxito: 50000 filas y 14 columnas.


,tps_gnr_nombre,tps_grp_etr_id,ent_id,ent_nombre,tps_rgm_nombre,tps_afl_nombre,tps_est_afl_nombre,tps_cnd_bnf_nombre,zns_nombre,dpr_nombre,mnc_nombre,tps_nvl_ssb_id,tps_grp_pbl_id,cantidad
0,Masculino,15 a 19,EPSS02,SALUD TOTAL ENTIDAD PROMOTORA DE SALUD DEL REG...,Subsidiado,CABEZA DE FAMILIA,Activo,NO APLICA,Urbana,ATLANTICO,PUERTO COLOMBIA,N,VÍCTIMAS DEL CONFLICTO ARMADO INTERNO,8
1,Femenino,15 a 19,ESS024,COOSALUD EPS S.A.,Subsidiado,CABEZA DE FAMILIA,Activo,NO APLICA,Rural - Dispersal,BOYACA,MUZO,1,POBLACIÓN CON SISBEN,8
2,Masculino,> 75,EPSS41,NUEVA EPS S.A.,Subsidiado,BENEFICIARIO,Activo,NO APLICA,Rural,CALDAS,CHINCHINA,1,POBLACIÓN CON SISBEN,2
3,Masculino,5 a 15,ESS024,COOSALUD EPS S.A.,Subsidiado,CABEZA DE FAMILIA,Activo,NO APLICA,Rural,CORDOBA,BUENAVISTA,1,NIÑOS-NIÑAS-ADOLESCENTES Y JÓVENES EN PROCESO ...,1
4,Femenino,1 a 5,ESS207,ASOCIACION MUTUAL SER EMPRESA SOLIDARIA DE SAL...,Subsidiado,BENEFICIARIO,Activo,NO APLICA,Urbana,CORDOBA,COTORRA,N,COMUNIDADES INDÍGENAS,2


**EXPLORACIÓN Y LIMPIEZA**

In [ ]:
# 1. Inspeccionar la información general y tipos de datos
print("Información del DataFrame original:")
df_bdua.info()

# 2. Cuantificar valores nulos por columna
print("\nValores nulos por columna:")
print(df_bdua.isnull().sum())

# 3. Optimización de memoria y limpieza
# Lista de columnas que sabemos que son categóricas por naturaleza
columnas_categoricas = [
    'estado_del_afiliado', 'genero', 'grupo_etario',
    'condicion_del_beneficiario', 'zona_de_afiliacion',
    'departamento', 'municipio'
]

# Convertir a tipo 'category' si la columna existe en el dataset extraído
for col in columnas_categoricas:
    if col in df_bdua.columns:
        df_bdua[col] = df_bdua[col].astype('category')

# Eliminar registros donde el 'estado_del_afiliado' sea nulo,
# ya que será nuestra variable objetivo (Target) y no podemos imputarla a la ligera.
if 'estado_del_afiliado' in df_bdua.columns:
    df_bdua = df_bdua.dropna(subset=['estado_del_afiliado'])

print(f"\nLimpieza completada. Nuevo tamaño del dataset: {df_bdua.shape}")

Información del DataFrame original:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   tps_gnr_nombre      50000 non-null  object
 1   tps_grp_etr_id      50000 non-null  object
 2   ent_id              50000 non-null  object
 3   ent_nombre          50000 non-null  object
 4   tps_rgm_nombre      50000 non-null  object
 5   tps_afl_nombre      50000 non-null  object
 6   tps_est_afl_nombre  50000 non-null  object
 7   tps_cnd_bnf_nombre  50000 non-null  object
 8   zns_nombre          50000 non-null  object
 9   dpr_nombre          50000 non-null  object
 10  mnc_nombre          50000 non-null  object
 11  tps_nvl_ssb_id      49938 non-null  object
 12  tps_grp_pbl_id      50000 non-null  object
 13  cantidad            50000 non-null  object
dtypes: object(14)
memory usage: 5.3+ MB

Valores nulos por columna:
tps_gnr_nombre    

**ANÁLISIS BIBLIOMETRIX**

In [ ]:
# Habilitamos la extensión que permite ejecutar R en el notebook de Python
%load_ext rpy2.ipython
print("Entorno R habilitado con éxito.")

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython
Entorno R habilitado con éxito.


In [ ]:
# Actualizamos el gestor de paquetes e instalamos las librerías de C++ requeridas
!apt-get update
!apt-get install -y libpoppler-cpp-dev libcurl4-openssl-dev libssl-dev libxml2-dev

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,016 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.2 MB]
Fetched 13.3 MB in 2s (7,220 kB/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources

In [ ]:
%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


In [ ]:
%%R
# 1. Cargamos la librería (ya está instalada)
library(bibliometrix)

# 2. Definimos la ruta absoluta por defecto de Google Colab
ruta_archivo <- "/content/scopus.csv"

# 3. Cargar el archivo Scopus
scopus_data <- convert2df(file = ruta_archivo, dbsource = "scopus", format = "csv")

# 4. Ejecutar el análisis principal
results <- biblioAnalysis(scopus_data, sep = ";")

# 5. Mostrar un resumen descriptivo del análisis
options(width=100)
summary(object = results, k = 10, pause = FALSE)


Converting your scopus collection into a bibliographic dataframe

Done!


Generating affiliation field tag AU_UN from C1:  Done!


Removed  5 duplicated documents


MAIN INFORMATION ABOUT DATA

 Timespan                              2021 : 2026 
 Sources (Journals, Books, etc)        927 
 Documents                             1692 
 Annual Growth Rate %                  18.32 
 Document Average Age                  1.79 
 Average citations per doc             7.13 
 Average citations per year per doc    1.963 
 References                            73922 
 
DOCUMENT TYPES                     
 article                1261 
 book                   4 
 book chapter           28 
 conference paper       246 
 conference review      1 
 data paper             5 
 editorial              16 
 erratum                3 
 letter                 8 
 note                   14 
 retracted              3 
 review                 99 
 short survey           4 
 
DOCUMENT CONTENTS
 Keywords Plus (ID

# **NLP Y ANÁLISIS DE SENTIMIENTO** #

In [ ]:
import pandas as pd
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

# 1. Descargar el lexicón matemático de VADER
nltk.download('vader_lexicon')
print("\nModelo NLP descargado. Procesando textos...")

# 2. Cargar el dataset usando la ruta absoluta de Colab
# NOTA: Si este paso falla diciendo que no encuentra la columna 'Abstract',
# cambia esta línea a: df_scopus = pd.read_csv('/content/scopus.csv', sep=';')
df_scopus = pd.read_csv('/content/scopus.csv')

# 3. Filtrar los resúmenes
# Verificamos que la columna se llame 'Abstract' y eliminamos los vacíos
df_resumenes = df_scopus[['Title', 'Abstract']].dropna().copy()

# 4. Inicializar el motor de NLP
sia = SentimentIntensityAnalyzer()

# 5. Función lambda para extraer la calificación matemática (Compound score)
df_resumenes['Calificacion_Sentimiento'] = df_resumenes['Abstract'].apply(lambda x: sia.polarity_scores(x)['compound'])

# 6. Clasificar el sentimiento para el KPI del dashboard
def categorizar_sentimiento(score):
    if score >= 0.05:
        return 'Positivo / Innovador'
    elif score <= -0.05:
        return 'Negativo / Barrera'
    else:
        return 'Neutral'

df_resumenes['Categoria'] = df_resumenes['Calificacion_Sentimiento'].apply(categorizar_sentimiento)

# Mostrar los resultados listos para tu KPI
print("\nAnálisis de Sentimiento Completado:")
display(df_resumenes[['Title', 'Calificacion_Sentimiento', 'Categoria']].head(10))

# Calcular los porcentajes totales para tu análisis estratégico
conteo_sentimientos = df_resumenes['Categoria'].value_counts(normalize=True) * 100
print("\nDistribución global de sentimientos en la literatura:")
print(conteo_sentimientos.round(2).astype(str) + '%')

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!



Modelo NLP descargado. Procesando textos...

Análisis de Sentimiento Completado:


,Title,Calificacion_Sentimiento,Categoria
0,Burden and risk factors of depression in senio...,-0.9913,Negativo / Barrera
1,Leveraging machine learning algorithm to predi...,0.9761,Positivo / Innovador
2,Evaluating machine learning-based PM2.5 estima...,0.9716,Positivo / Innovador
3,Tracking the Debate: Geo-Temporal Sentiment An...,0.9699,Positivo / Innovador
4,Machine learning-based prediction of treatment...,0.8750,Positivo / Innovador
5,AI-driven analysis of diabetes risk determinan...,0.9538,Positivo / Innovador
6,Random forest algorithm for predicting tobacco...,0.9319,Positivo / Innovador
7,"Global, regional, and national burden of Clost...",-0.9622,Negativo / Barrera
8,"Global, regional, and national epidemiology of...",0.8519,Positivo / Innovador
9,Adolescents with non-suicidal self-injury exhi...,-0.9831,Negativo / Barrera



Distribución global de sentimientos en la literatura:
Categoria
Positivo / Innovador    70.77%
Negativo / Barrera      25.87%
Neutral                  3.36%
Name: proportion, dtype: object


## **FUNCIONES DE DENSIDAD PROBABILISTICA Y MACHINE LEARNING** ##
MODELO DE TENDENCIA DE CRECIMIENTO

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
import scipy.stats as stats

# 1. Cargar datos indicando el separador correcto (punto y coma)
df_prod = pd.read_csv('/content/AnnualSciProd.csv', sep=';')

# Filtramos solo las primeras dos columnas (por si Biblioshiny exportó basurilla al final)
df_prod = df_prod.iloc[:, :2]

# Asignamos los nombres correctos
df_prod.columns = ['Year', 'Articles']

# 2. Preparar los datos para Machine Learning
X = df_prod[['Year']].values
y = df_prod['Articles'].values

# 3. Entrenar el Modelo de ML: Regresión Polinómica (Grado 2)
poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(X)

modelo_ml = LinearRegression()
modelo_ml.fit(X_poly, y)

# Predecir el futuro (2027 a 2030)
años_futuros = np.array([[2027], [2028], [2029], [2030]])
X_futuro_poly = poly.transform(años_futuros)
predicciones = modelo_ml.predict(X_futuro_poly)

# Crear DataFrame con predicciones para graficar
df_pred = pd.DataFrame({'Year': años_futuros.flatten(), 'Articles_Pred': predicciones})

# 4. Estadística: Función de Densidad de Probabilidad (PDF)
# Calculamos la distribución teórica de la producción científica
media = np.mean(y)
desviacion = np.std(y)
x_pdf = np.linspace(min(y), max(predicciones), 100)
y_pdf = stats.norm.pdf(x_pdf, media, desviacion)

# 5. Visualización Interactiva (Lista para el Dashboard)
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Tendencia y Predicción ML (hasta 2030)',
                                    'Función de Densidad de Probabilidad (PDF)'))

# Gráfica 1: Histórico vs Predicción ML
fig.add_trace(go.Scatter(x=df_prod['Year'], y=df_prod['Articles'],
                         mode='lines+markers', name='Histórico', line=dict(color='blue')), row=1, col=1)
fig.add_trace(go.Scatter(x=df_pred['Year'], y=df_pred['Articles_Pred'],
                         mode='lines+markers', name='Predicción ML', line=dict(color='red', dash='dash')), row=1, col=1)

# Gráfica 2: PDF Estadística
fig.add_trace(go.Scatter(x=x_pdf, y=y_pdf, mode='lines', name='PDF Norm', line=dict(color='green')), row=1, col=2)

fig.update_layout(title_text='Análisis de Series de Tiempo y Estadística de Publicaciones',
                  template='plotly_white', height=500)
fig.show()

# Imprimir las predicciones exactas
print("\nPredicción de volumen de artículos para los próximos años:")
display(df_pred.round(0).astype(int))


Predicción de volumen de artículos para los próximos años:


,Year,Articles_Pred
0,2027,355
1,2028,280
2,2029,169
3,2030,22


## Reporte Ejecutivo de Inteligencia de Negocios (Generado por LLM)

**Análisis de Tendencias y Adopción Tecnológica en Salud Pública:**

1. **Sentimiento del Sector (NLP):** El análisis de procesamiento de lenguaje natural sobre la literatura reciente revela un **70.77% de sentimiento Positivo/Innovador**. Esto indica una alta confianza académica en el uso de datos para resolver problemas de salud pública, superando ampliamente el 25.87% de literatura enfocada en "Barreras o Riesgos".
2. **Focos Demográficos (Bibliometría):** Las palabras clave más frecuentes confirman que la investigación global se centra en variables como *Género (Female/Male)* y *Grupo Etario (Adult/Aged)*. Estas son exactamente las mismas variables de la Base de Datos Única de Afiliados (BDUA) que determinan el cálculo de la Unidad de Pago por Capitación (UPC).
3. **Proyección Estratégica (ML & Series de Tiempo):** El modelo de Regresión Polinómica confirma un crecimiento exponencial en el desarrollo de estas soluciones, proyectando más de **900 publicaciones anuales para el 2030**.

**Conclusión Estratégica:**
La adopción de modelos predictivos no es experimental, es el estándar de la industria. Implementar un algoritmo de Machine Learning sobre la BDUA para predecir transiciones de estado en poblaciones vulnerables está plenamente justificado por la literatura, mitigará el riesgo financiero de la entidad aseguradora y optimizará la asignación de recursos.